# 02 — Alpha Diversity

**Manuscript:** Maternal gut microbiome diversity and functional potential in early pregnancy are associated with large-for-gestational-age birth (SweMaMi cohort)

**Purpose:** Computes observed species richness, Shannon diversity and Pielou's evenness from MetaPhlAn4 relative abundances. Compares LGA vs AGA at TP1 and TP2 (unadjusted t-tests and confounder-adjusted logistic regression), and examines longitudinal within-subject change (TP1→TP2) using linear mixed-effects models.

**Produces:** Figure 2 (alpha diversity at TP1/TP2), Figure 3 (longitudinal change)


In [ ]:
library(dplyr)
library(tidyverse)
library(vegan)
library(ggplot2)
library(patchwork)
library("ggrain")
library(lme4)
library(lmerTest)

In [ ]:
base_path <- "data"

In [ ]:
all_taxa <- read.table(file.path(base_path,"taxa_tp1.tsv"),
  sep         = "\t",
  header      = TRUE,
  check.names = FALSE
)
head(all_taxa)

In [ ]:
new_row <- c()
tot <- colSums(all_taxa)
for(i in tot){
    if(i < 100) {
        new_row <- c(new_row,100 - i)
    } else if( i >=100){
        new_row <- c(new_row,0)
    }
}

new_row <- setNames(new_row, names(all_taxa))
new_row2 <- as.data.frame(t(new_row))
rownames(new_row2) <- "Other_taxa"
best_filt_new <- rbind(all_taxa,new_row2) 
filt_new_arrange_1 <- as.data.frame(best_filt_new)
dim(filt_new_arrange_1)
head(filt_new_arrange_1)

In [ ]:
all_taxa_2 <- read.table(file.path(base_path,"taxa_tp2.tsv"),
  sep         = "\t",
  header      = TRUE,
  check.names = FALSE
)
# Note: Other_taxa is already added

In [ ]:
meta_data_tp1 <- read.csv(file.path(base_path,"meta_data_tp1.csv"))
meta_data_tp2 <- read.csv(file.path(base_path, "meta_data_tp2.csv"))

In [ ]:
meta_data_tp1 <- meta_data_tp1 %>%
  mutate(kit1.faecal_sample.barcode = as.character(kit1.faecal_sample.barcode))
meta_data_tp2 <- meta_data_tp2 %>%
  mutate(kit2.faecal_sample.barcode = as.character(kit2.faecal_sample.barcode))


In [ ]:
meta_data_tp1$Primipara <- factor(meta_data_tp1$Primipara, levels = c("0","1"))
meta_data_tp1$excess_weight <- factor(meta_data_tp1$excess_weight, levels = c("No","Yes"))
meta_data_tp1$Gest_Diabetes <- factor(meta_data_tp1$Gest_Diabetes,levels = c("0","1"))
meta_data_tp1$Q1_healthydiet <- factor(meta_data_tp1$Q1_healthydiet, levels = c("1","0"))
meta_data_tp1$Q2_healthydiet <- factor(meta_data_tp1$Q2_healthydiet, levels = c("1","0"))
meta_data_tp1$Group <- factor(meta_data_tp1$Group, levels = c("Control","Case"))


In [ ]:
meta_data_tp2$Primipara <- factor(meta_data_tp2$Primipara, levels = c("0","1"))
meta_data_tp2$excess_weight <- factor(meta_data_tp2$excess_weight, levels = c("No","Yes"))
meta_data_tp2$Gest_Diabetes <- factor(meta_data_tp2$Gest_Diabetes,levels = c("0","1"))
meta_data_tp2$Q1_healthydiet <- factor(meta_data_tp2$Q1_healthydiet, levels = c("1","0"))
meta_data_tp2$Q2_healthydiet <- factor(meta_data_tp2$Q2_healthydiet, levels = c("1","0"))
meta_data_tp2$Group <- factor(meta_data_tp2$Group, levels = c("Control","Case"))


## 1. Calculate , Shannon,richness, evenness (TP1, TP2)


### TP1

In [ ]:
case_df_1 <- meta_data_tp1 %>%
  filter(Group == "Case")

  case_kit1 <- case_df_1  %>%
  pull("kit1.faecal_sample.barcode")

In [ ]:
control_df_1 <- meta_data_tp1 %>%
  filter(Group == "Control")

   control_kit1 <- control_df_1 %>%
  pull("kit1.faecal_sample.barcode")
 

In [ ]:

Case_taxa_1 <- filt_new_arrange_1[,colnames(filt_new_arrange_1) %in% case_kit1]
dim(Case_taxa_1)

Control_taxa_1 <- filt_new_arrange_1[,colnames(filt_new_arrange_1) %in% control_kit1]
dim(Control_taxa_1)


In [ ]:
#Shannon
case_shan_1 <- apply(Case_taxa_1, 2, diversity, index = "shannon")

control_shan_1 <- apply(Control_taxa_1, 2, diversity, index = "shannon")


In [ ]:
#calculates species richness 
case_rich_1 <- apply(Case_taxa_1, 2, specnumber)
control_rich_1 <- apply(Control_taxa_1, 2, specnumber)

In [ ]:
#Evenness: Pielou
case_even_1 <- case_shan_1/log(case_rich_1)
control_even_1 <- control_shan_1/log(case_rich_1)

### TP2

In [ ]:
case_df_2 <- meta_data_tp2 %>%
  filter(Group == "Case")

  case_kit2 <- case_df_2  %>%
  pull("kit2.faecal_sample.barcode")


In [ ]:
control_df_2 <- meta_data_tp2 %>%
  filter(Group == "Control")

   control_kit2 <- control_df_2 %>%
  pull("kit2.faecal_sample.barcode")


In [ ]:
Case_taxa_2 <- all_taxa_2[,colnames(all_taxa_2) %in% case_kit2 ]
dim(Case_taxa_2)

Control_taxa_2 <- all_taxa_2[,colnames(all_taxa_2) %in% control_kit2]
dim(Control_taxa_2)


In [ ]:
#Shannon
case_shan_2 <- apply(Case_taxa_2, 2, diversity, index = "shannon")

control_shan_2 <- apply(Control_taxa_2, 2, diversity, index = "shannon")


In [ ]:
#calculates species richness 
case_rich_2 <- apply(Case_taxa_2, 2, specnumber)
control_rich_2 <- apply(Control_taxa_2, 2, specnumber)

In [ ]:
#Evenness: Pielou
case_even_2 <- case_shan_2/log(case_rich_2)
control_even_2 <- control_shan_2/log(control_rich_2)

### TP1

In [ ]:
#Shannon diversity
tp1_shan_df <- data.frame(shannon = c(case_shan_1 , control_shan_1),
                 Group = rep(c("Case", "Control"), times = c(length(case_shan_1), length(control_shan_1))))
head(tp1_shan_df)


In [ ]:
# Histogram for case_shan_1
hist(case_shan_1, main = "Histogram of Case Group", xlab = "Values", col = "deeppink", breaks = 10)

# Histogram for control_shan_1
hist(control_shan_1, main = "Histogram of Control Group", xlab = "Values", col = "#40E0D0", breaks = 10)

In [ ]:
t.test(shannon ~ Group, data = tp1_shan_df)
p_value_shan_1 <- t.test(shannon ~ Group, data = tp1_shan_df)$p.value
p_value_shan_1

In [ ]:
#Richness
tp1_rich_df <- data.frame(Richness = c(case_rich_1,control_rich_1),
                 Group = rep(c("Case", "Control"), times = c(length(case_rich_1), length(control_rich_1))))
head(tp1_rich_df)

In [ ]:
# Histogram for case_rich_1
hist(case_rich_1, main = "Histogram of Case Group (Richness)", xlab = "Values", col = "deeppink", breaks = 10)

# Histogram for control_rich_1
hist(control_rich_1, main = "Histogram of Control Group (Richness)", xlab = "Values", col = "#40E0D0", breaks = 10)


In [ ]:
t.test(Richness ~ Group, data = tp1_rich_df)
p_value_rich_1 <- t.test(Richness ~ Group, data = tp1_rich_df)$p.value
p_value_rich_1

In [ ]:
#Evenness
evenness_tp1 <- data.frame(Evenness = c(case_even_1,control_even_1),
                 Group = rep(c("Case", "Control"), times = c(length(case_even_1), length(control_even_1))))
head(evenness_tp1)

In [ ]:
t.test(Evenness ~ Group, data = evenness_tp1)
p_value_even_1 <- t.test(Evenness ~ Group, data = evenness_tp1)$p.value
p_value_even_1

### TP2

In [ ]:
#Shannon diversity
tp2_shan_df <- data.frame(shannon = c(case_shan_2 , control_shan_2),
                 Group = rep(c("Case", "Control"), times = c(length(case_shan_2), length(control_shan_2))))
head(tp2_shan_df)

In [ ]:
# Histogram for case_shan_2
hist(case_shan_2, main = "Histogram of Case Group", xlab = "Values", col = "#E30B5D", breaks = 10)

# Histogram for control_shan_2
hist(control_shan_2, main = "Histogram of Control Group", xlab = "Values", col = "#708090", breaks = 10)

In [ ]:
t.test(shannon ~ Group, data = tp2_shan_df)
p_value_shan_2 <- t.test(shannon ~ Group, data = tp2_shan_df)$p.value
p_value_shan_2

In [ ]:
#Richness
tp2_rich_df <- data.frame(Richness = c(case_rich_2,control_rich_2),
                 Group = rep(c("Case", "Control"), times = c(length(case_rich_2), length(control_rich_2))))
head(tp2_rich_df)

In [ ]:
t.test(Richness ~ Group, data = tp2_rich_df)
p_value_rich_2 <- t.test(Richness ~ Group, data = tp2_rich_df)$p.value
p_value_rich_2

In [ ]:
#Evenness
evenness_tp2 <- data.frame(Evenness = c(case_even_2,control_even_2),
                 Group = rep(c("Case", "Control"), times = c(length(case_even_2), length(control_even_2))))
head(evenness_tp2)

In [ ]:
t.test(Evenness ~ Group, data = evenness_tp2)
p_value_even_2 <- t.test(Evenness ~ Group, data = evenness_tp2)$p.value
p_value_even_2

## 2. Adjusted logistic regression per metric


### TP1

In [ ]:
tp1_shan_df <- tp1_shan_df %>%
  rownames_to_column(var = "kit1.faecal_sample.barcode")

tp1_rich_df <- tp1_rich_df %>%
  rownames_to_column(var = "kit1.faecal_sample.barcode")

evenness_tp1 <- evenness_tp1 %>%
  rownames_to_column(var = "kit1.faecal_sample.barcode")

In [ ]:
tp1_shan_df <- tp1_shan_df %>%
  mutate(kit1.faecal_sample.barcode = as.character(kit1.faecal_sample.barcode))

tp1_rich_df <- tp1_rich_df %>%
  mutate(kit1.faecal_sample.barcode = as.character(kit1.faecal_sample.barcode))

evenness_tp1 <- evenness_tp1 %>%
  mutate(kit1.faecal_sample.barcode = as.character(kit1.faecal_sample.barcode))

In [ ]:
# add the diversity metric column to the metadata

tp1_meta_alpha <- meta_data_tp1 %>%
  left_join(tp1_shan_df %>% select(-Group), by = "kit1.faecal_sample.barcode") %>%
  left_join(tp1_rich_df %>% select(-Group), by = "kit1.faecal_sample.barcode") %>%
  left_join(evenness_tp1 %>% select(-Group), by = "kit1.faecal_sample.barcode")

In [ ]:
#Shannon diversity adjusting for the potential confounders
model_shan_tp1 <- glm(Group ~ shannon + age_checked + BMI_prior + Primipara  + Q1_healthydiet,
                   data = tp1_meta_alpha,
                   family = binomial)
summary(model_shan_tp1)

In [ ]:
#Richness adjusting for the potential confounders
model_rich_tp1 <- glm(Group ~ Richness + age_checked + BMI_prior + Primipara  + Q1_healthydiet ,
                   data = tp1_meta_alpha ,
                   family = binomial)
summary(model_rich_tp1)

In [ ]:
#Evenness adjusting for the potential confounders
model_even_tp1 <- glm(Group ~  Evenness + age_checked + BMI_prior + Primipara  + Q1_healthydiet,
                   data = tp1_meta_alpha ,
                   family = binomial)
summary(model_even_tp1)

### TP2

In [ ]:
tp2_shan_df <- tp2_shan_df %>%
  rownames_to_column(var = "kit2.faecal_sample.barcode")

tp2_rich_df <- tp2_rich_df %>%
  rownames_to_column(var = "kit2.faecal_sample.barcode")

evenness_tp2 <- evenness_tp2 %>%
  rownames_to_column(var = "kit2.faecal_sample.barcode")

In [ ]:
tp2_shan_df <- tp2_shan_df %>%
  mutate(kit2.faecal_sample.barcode = as.character(kit2.faecal_sample.barcode))

tp2_rich_df <- tp2_rich_df %>%
  mutate(kit2.faecal_sample.barcode = as.character(kit2.faecal_sample.barcode))

evenness_tp2 <- evenness_tp2 %>%
  mutate(kit2.faecal_sample.barcode = as.character(kit2.faecal_sample.barcode))

In [ ]:
# add the diversity metric column to the metadata

tp2_meta_alpha <- meta_data_tp2 %>%
  left_join(tp2_shan_df %>% select(-Group), by = "kit2.faecal_sample.barcode") %>%
  left_join(tp2_rich_df %>% select(-Group), by = "kit2.faecal_sample.barcode") %>%
  left_join(evenness_tp2 %>% select(-Group), by = "kit2.faecal_sample.barcode")

In [ ]:
#Shannon diversity adjusting for the potential confounders
model_shan_tp2 <- glm(Group ~ shannon + age_checked + BMI_prior + Primipara  + Q2_healthydiet,
                   data = tp2_meta_alpha,
                   family = binomial)
summary(model_shan_tp2)

In [ ]:
#Richness adjusting for the potential confounders
model_rich_tp2 <- glm(Group ~ Richness + age_checked + BMI_prior + Primipara  + Q2_healthydiet ,
                   data = tp2_meta_alpha ,
                   family = binomial)
summary(model_rich_tp2)

In [ ]:
#Evenness adjusting for the potential confounders
model_even_tp2 <- glm(Group ~  Evenness + age_checked + BMI_prior + Primipara  + Q2_healthydiet,
                   data = tp2_meta_alpha ,
                   family = binomial)
summary(model_even_tp2)

In [ ]:
# Function to extract OR, CI, and p-value 
extract_stats <- function(model, exposure_name) {
  coef_summary <- summary(model)$coefficients
  est <- coef_summary[exposure_name, "Estimate"]
  se <- coef_summary[exposure_name, "Std. Error"]
  pval <- coef_summary[exposure_name, "Pr(>|z|)"]
  
  # Calculate OR and 95% CI
  OR <- exp(est)
  CI_lower <- exp(est - 1.96 * se)
  CI_upper <- exp(est + 1.96 * se)
  
  # Create p_group column
  p_group <- factor(ifelse(pval > 0.05, "p > 0.05", "p <= 0.05"),
                    levels = c("p <= 0.05", "p > 0.05"))
  

  
  return(data.frame(
    Alpha_diversity = exposure_name,
    OR = OR,
    CI_lower = CI_lower,
    CI_upper = CI_upper,
    p_value = pval,
    p_group
  ))
}

## 3. Figure 2

### TP1

In [ ]:
#Shannon diversity Violin plot
my_plot_shan <- ggplot(tp1_shan_df,aes(x= Group, y= shannon, fill = Group))+
                geom_violin(trim = FALSE)+
                geom_boxplot(width = 0.1, fill = "white", outlier.shape = NA)+
                scale_fill_manual(values = c("Control" = "#40E0D0",
                             "Case" = "deeppink"))+
                labs(title = "Shannon Diversity Index", x = "Group", y= "Shannon Index")+
                theme_minimal() +
                theme(axis.title.x = element_text(size = 14, face = "bold"),
                axis.text.x = element_text(size = 12,face = "bold"),
                axis.title.y = element_text(size = 12, face = "bold"),
                axis.text.y = element_text(size = 12,face = "bold"),
                plot.title = element_text(size = 14, face = "bold"),
                legend.title = element_text(size = 12,face = "bold"), 
                legend.text = element_text(size = 12))
#my_plot_shan
plot_p_shan <- my_plot_shan +
  annotate("text", x = 1.3, y = 4.99,
           label = paste("p =", round(p_value_shan_1, 3)),
           color = "black", size = 5.0, hjust = 0) +
  geom_segment(aes(x = 1, xend = 2, y = 4.9, yend = 4.9),
               color = "black", size = 0.4)
plot_p_shan



In [ ]:
#Richness diversity Violin plot
my_plot_rich <- ggplot(tp1_rich_df,aes(x= Group, y= Richness, fill = Group))+
                geom_violin(trim = FALSE)+
                geom_boxplot(width = 0.1, fill = "white", outlier.shape = NA)+
                scale_fill_manual(values = c("Case" = "deeppink",
                                            "Control" = "#40E0D0"))+
                labs(title = "Observed species", x = "Group", y= "Richness")+
                theme_minimal() +
                theme(axis.title.x = element_text(size = 14, face = "bold"),
                axis.text.x = element_text(size = 12,face = "bold"),
                axis.title.y = element_text(size = 12, face = "bold"),
                axis.text.y = element_text(size = 12,face = "bold"),
                plot.title = element_text(size = 14, face = "bold"),
                legend.title = element_text(size = 12,face = "bold"), 
                legend.text = element_text(size = 12))
my_plot_rich_s <- my_plot_rich +
  annotate("text", x = 1.4, y = 583,
           label = paste("p =", round(p_value_rich_1, 3)),
           color = "black", size = 5.0, hjust = 0) +
  geom_segment(aes(x = 1, xend = 2, y = 570, yend = 570),
               color = "black", size = 0.4)
my_plot_rich_s


In [ ]:
#Evenness Violin plot
my_plot_even <- ggplot(evenness_tp1,aes(x= Group, y= Evenness, fill = Group))+
                geom_violin(trim = FALSE)+
                geom_boxplot(width = 0.1, fill = "white", outlier.shape = NA)+
                scale_fill_manual(values = c("Case" = "deeppink",
                                            "Control" = "#40E0D0"))+
                labs(title = "Pielou's evenness", x = "Group", y= "Evenness")+
                theme_minimal() +
                theme(axis.title.x = element_text(size = 14, face = "bold"),
                axis.text.x = element_text(size = 12,face = "bold"),
                axis.title.y = element_text(size = 12, face = "bold"),
                axis.text.y = element_text(size = 12,face = "bold"),
                plot.title = element_text(size = 14,face = "bold"),
                legend.title = element_text(size = 12,face = "bold"), 
                legend.text = element_text(size = 12))
my_plot_even_s <- my_plot_even +
  annotate("text", x = 1.30, y = 0.83,
           label = paste("p =", round(p_value_even_1, 3)),
           color = "black", size = 5.0, hjust = 0) +
  geom_segment(aes(x = 1, xend = 2, y = 0.82, yend = 0.82),
               color = "black", size = 0.4)
my_plot_even_s


In [ ]:
plot_shan_1 <- plot_p_shan
plot_rich_1 <- my_plot_rich_s
plot_even_1 <- my_plot_even_s

In [ ]:
final_plot_raw_alpha_tp1 <- (plot_shan_1 + plot_rich_1 + plot_even_1) + 
  plot_layout(ncol = 3, guides = "collect") +
  plot_annotation(
    title = "A) Alpha Diversity at TP1",
    theme = theme(plot.title = element_text(size = 16, face = "bold"))
  ) & 
  theme(legend.position = "right",
        text = element_text(size = 12))

final_plot_raw_alpha_tp1 

In [ ]:
shan_stats_tp1 <- extract_stats(model_shan_tp1, "shannon")
rich_stats_tp1 <- extract_stats(model_rich_tp1, "Richness")
even_stats_tp1 <- extract_stats(model_even_tp1, "Evenness")

In [ ]:
forest_data_tp1 <- rbind(shan_stats_tp1, rich_stats_tp1, even_stats_tp1)
# To arrange the order in the plot. 
forest_data_tp1$Alpha_diversity <- factor(
forest_data_tp1$Alpha_diversity,
  levels = forest_data_tp1$Alpha_diversity
)
forest_data_tp1$Alpha_diversity <- dplyr::recode(
 forest_data_tp1$Alpha_diversity,
  "shannon" = "Shannon Index",
  "Richness" = "Richness",
  "Evenness" = "Evenness"
)
head(forest_data_tp1)

In [ ]:
#  Prepare data
# All three metrics — same method, same scale
forest_data_tp11 <- forest_data_tp1 %>%
  mutate(
    pval_label = case_when(
      p_value < 0.001 ~ "p < 0.001",
      p_value < 0.01  ~ paste0("p = ",
                                round(p_value, 3)),
      TRUE            ~ paste0("p = ",
                                round(p_value, 3))
    ),
    or_label = paste0(
      round(OR, 2),
      "\n(", round(CI_lower, 2),
      "–", round(CI_upper, 2), ")"
    )
  )
forest_data_tp11

In [ ]:
forest_data_tp11 %>%
  filter(Alpha_diversity == "Shannon Index") %>%
  distinct(p_group)
forest_data_tp11

In [ ]:
make_clean_panel <- function(data,
                             title,
                             x_limits,
                             x_breaks,
                             use_log = FALSE) {
  p <- ggplot(data,
              aes(x     = OR,
                  y     = Alpha_diversity,
                  color = p_group)) +
    
    geom_vline(xintercept = 1,
               linetype   = "dashed",
               color      = "#999999",
               linewidth  = 0.8) +
    
    geom_errorbarh(aes(xmin = CI_lower,
                       xmax = CI_upper),
                   height    = 0.15,
                   linewidth = 1.0) +
    
    geom_point(size = 6, shape = 18) +
    
    # P-value now BOLD and black
    geom_text(aes(x     = OR,
                  label = pval_label),
              vjust    = -1.5,
              size     = 4,
              color    = "#000000") +
    
    scale_color_manual(
      values = c("p <= 0.05" = "#000000FF",
                 "p > 0.05"  = "#FF0000"),
      name   = ""
    ) +
    
    coord_cartesian(xlim = x_limits) +
    
    theme_minimal(base_size = 12) +
    theme(
      panel.grid.major.y = element_blank(),
      panel.grid.minor   = element_blank(),
      legend.position    = "none",
      # all text forced to black
      text               = element_text(color = "#000000"),
      axis.title.x       = element_text(size = 10, face = "bold",
                                        color = "#000000"),
      axis.text.y        = element_text(size = 12, face = "bold",
                                        color = "#000000"),
      axis.text.x        = element_text(size = 9, color = "#000000"),
      plot.title         = element_text(size = 12, face = "bold",
                                        hjust = 0.5, color = "#000000"),
      panel.border       = element_rect(color     = "#dddddd",
                                        fill      = NA,
                                        linewidth = 0.5)
    ) +
    labs(title = title,
         x     = ifelse(use_log,
                        "OR (95% CI) — Log Scale",
                        "OR (95% CI)"),
         y     = "")
  
  if (use_log) {
    p <- p +
      scale_x_log10(breaks = x_breaks,
                    labels = as.character(x_breaks))
  } else {
    p <- p +
      scale_x_continuous(breaks = x_breaks)
  }
  
  return(p)
}

In [ ]:
# Panel 1: Shannon
p1 <- make_clean_panel(
  data     = forest_data_tp11 %>%
               filter(Alpha_diversity == "Shannon Index"),
  title    = "Shannon Diversity Index",
  x_limits = c(0.5, 5),
  x_breaks = c(0.5, 1.0, 1.5, 2.0, 3.0),
  use_log  = FALSE
)

# Panel 2: Richness
p2 <- make_clean_panel(
  data     = forest_data_tp11 %>%
               filter(Alpha_diversity == "Richness"),
  title    = "Observed Species",
  x_limits = c(0.999, 1.008),
  x_breaks = c(1.000, 1.002,
                1.004, 1.006),
  use_log  = FALSE
)


# Panel 3: Evenness (log scale due to wide CI)
p3 <- make_clean_panel(
  data     = forest_data_tp11 %>%
               filter(Alpha_diversity == "Evenness"),
  title    = "Pielou's evenness",
  x_limits = c(0.03, 300),
  x_breaks = c(0.1, 1, 10, 100),
  use_log  = TRUE   # log scale for wide CI
)


In [ ]:
#Three panels stacked vertically
final_plot_tp1 <- p1 / p2 / p3 +   
  plot_annotation(
    title    = "C) Adjusted Odds Ratios — Alpha Diversity at TP1",
    theme    = theme(
      plot.title    = element_text(size = 14, face = "bold"),
      plot.subtitle = element_text(size = 11, color = "#555")
    )
  )

print(final_plot_tp1)

### TP2

In [ ]:
##Shannon diversity Violin plot
my_plot_shan_2 <- ggplot(tp2_shan_df,aes(x= Group, y= shannon, fill = Group))+
                geom_violin(trim = FALSE)+
                geom_boxplot(width = 0.1, fill = "white", outlier.shape = NA)+
                scale_fill_manual(values = c("Control" = "#708090",
                             "Case" = "#E30B5D"))+
                labs(title = "Shannon Diversity Index", x = "Group", y= "Shannon Index")+
                theme_minimal() +
                theme(axis.title.x = element_text(size = 12, face = "bold"),
                axis.text.x = element_text(size = 12,face = "bold"),
                axis.title.y = element_text(size = 12, face = "bold"),
                axis.text.y = element_text(size = 12,face = "bold"),
                plot.title = element_text(size = 12, face = "bold"),
                legend.title = element_text(size = 12,face = "bold"), 
                legend.text = element_text(size = 12))
#my_plot_shan
plot_p_shan_2 <- my_plot_shan_2 +
  annotate("text", x = 1.38, y = 5.08,
           label = paste("p =", round(p_value_shan_2, 3)),
           color = "black", size = 4.0, hjust = 0) +
  geom_segment(aes(x = 1, xend = 2, y = 5.01, yend = 5.01),
               color = "black", size = 0.4)
plot_p_shan_2


In [ ]:
#Richness diversity Violin plot
my_plot_rich_2 <- ggplot(tp2_rich_df,aes(x= Group, y= Richness, fill = Group))+
                geom_violin(trim = FALSE)+
                geom_boxplot(width = 0.1, fill = "white", outlier.shape = NA)+
                scale_fill_manual(values = c("Case" = "#E30B5D",
                                            "Control" = "#708090"))+
                labs(title = "Observed species", x = "Group", y= "Richness")+
                theme_minimal() +
                theme(axis.title.x = element_text(size = 12, face = "bold"),
                axis.text.x = element_text(size = 12,face = "bold"),
                axis.title.y = element_text(size = 12, face = "bold"),
                axis.text.y = element_text(size = 12,face = "bold"),
                plot.title = element_text(size = 12, face = "bold"),
                legend.title = element_text(size = 12,face = "bold"), 
                legend.text = element_text(size = 12))
my_plot_rich_s_2 <- my_plot_rich_2 +
  annotate("text", x = 1.4, y = 583,
           label = paste("p =", round(p_value_rich_2, 3)),
           color = "black", size = 4.0, hjust = 0) +                      # ← match forest plots
  geom_segment(aes(x = 1, xend = 2, y = 570, yend = 570),
               color = "black", size = 0.4)

my_plot_rich_s_2


In [ ]:
#Evenness Violin plot
my_plot_even_2 <- ggplot(evenness_tp2,aes(x= Group, y= Evenness, fill = Group))+
                geom_violin(trim = FALSE)+
                geom_boxplot(width = 0.1, fill = "white", outlier.shape = NA)+
                scale_fill_manual(values = c("Case" = "#E30B5D",
                                            "Control" = "#708090"))+
                labs(title = "Pielou's evenness", x = "Group", y= "Evenness")+
                theme_minimal() +
                theme(axis.title.x = element_text(size = 12, face = "bold"),
                axis.text.x = element_text(size = 12,face = "bold"),
                axis.title.y = element_text(size = 12, face = "bold"),
                axis.text.y = element_text(size = 12,face = "bold"),
                plot.title = element_text(size = 12, face = "bold"),
                legend.title = element_text(size = 12,face = "bold"), 
                legend.text = element_text(size = 12))
my_plot_even_s_2 <- my_plot_even_2 +
  annotate("text", x = 1.30, y = 0.83,
           label = paste("p =", round(p_value_even_2, 3)),
           color = "black", size = 4.0, hjust = 0) +
  geom_segment(aes(x = 1, xend = 2, y = 0.82, yend = 0.82),
               color = "black", size = 0.4)
my_plot_even_s_2


In [ ]:
plot_shan_2 <- plot_p_shan_2 
plot_rich_2 <- my_plot_rich_s_2
plot_even_2 <- my_plot_even_s_2


In [ ]:
final_plot_raw_alpha_tp2 <- (plot_shan_2 + plot_rich_2 + plot_even_2) + 
  plot_layout(ncol = 3, guides = "collect") +
  plot_annotation(
    title = "B) Alpha Diversity at TP2",
    theme = theme(plot.title = element_text(size = 14, face = "bold"))
  ) & 
  theme(legend.position = "right",
        text = element_text(size = 12))

final_plot_raw_alpha_tp2 

In [ ]:
shan_stats_tp2 <- extract_stats(model_shan_tp2, "shannon")
rich_stats_tp2 <- extract_stats(model_rich_tp2, "Richness")
even_stats_tp2 <- extract_stats(model_even_tp2, "Evenness")

In [ ]:
forest_data_tp2 <- rbind(shan_stats_tp2, rich_stats_tp2, even_stats_tp2)
forest_data_tp2$Alpha_diversity <- factor(
forest_data_tp2$Alpha_diversity,
  levels = forest_data_tp2$Alpha_diversity
)
forest_data_tp2$Alpha_diversity <- dplyr::recode(
 forest_data_tp2$Alpha_diversity,
  "shannon" = "Shannon Index",
  "Richness" = "Richness",
  "Eveness" = "Evenness"
)
head(forest_data_tp2)

In [ ]:
# All three metrics — same method, same scale
forest_data_tp22 <- forest_data_tp2 %>%
  mutate(
    pval_label = case_when(
      p_value < 0.001 ~ "p < 0.001",
      p_value < 0.01  ~ paste0("p = ",
                                round(p_value, 3)),
      TRUE            ~ paste0("p = ",
                                round(p_value, 3))
    ),
    or_label = paste0(
      round(OR, 2),
      "\n(", round(CI_lower, 2),
      "–", round(CI_upper, 2), ")"
    )
  )
forest_data_tp22

In [ ]:
# Panel 1: Shannon
tp2_p1 <- make_clean_panel(
  data     = forest_data_tp22 %>%
               filter(Alpha_diversity == "Shannon Index"),  
  title    = "Shannon Diversity Index",
  x_limits = c(0.4, 3),     # adjusted for OR = 1.06
  x_breaks = c(0.5, 1.0, 1.5, 2.0, 2.5),
  use_log  = FALSE
)


In [ ]:
# Panel 2: Richness
tp2_p2 <- make_clean_panel(
  data     = forest_data_tp22 %>%
               filter(Alpha_diversity == "Richness"),
  title    = "Observed Species",
  x_limits = c(0.998, 1.007),
  x_breaks = c(0.999, 1.000, 1.002,
                1.004, 1.006),
  use_log  = FALSE
)


In [ ]:
# Panel 3: Evenness (log scale — wide CI)
tp2_p3 <- make_clean_panel(
  data     = forest_data_tp22 %>%
               filter(Alpha_diversity == "Evenness"),
  title    = "Pielou's evenness",
  x_limits = c(0.001, 50),
  x_breaks = c(0.01, 0.1, 1, 10),
  use_log  = TRUE
)


In [ ]:
# Combine all the tp2 forest plots
final_tp2 <- tp2_p1 / tp2_p2 / tp2_p3 +
  plot_annotation(
    title    = "D) Adjusted Odds Ratios — Alpha Diversity at TP2",
    theme    = theme(
      plot.title    = element_text(size = 14,
                                    face = "bold"),
      plot.subtitle = element_text(size = 11,
                                    color = "#555")
    )
  )



### Figure -2

In [ ]:
figure_2 <-
  (final_plot_raw_alpha_tp1 | final_plot_raw_alpha_tp2) /
  (final_plot_tp1 | final_tp2) +
  plot_layout(heights = c(1, 1.6)) +
  plot_annotation(
    tag_levels = list(c("A) Alpha Diversity at TP1",
                        "B) Alpha Diversity at TP2",
                        "C) Adjusted Odds Ratios — TP1",
                        "D) Adjusted Odds Ratios — TP2"))
  ) &
  theme(plot.tag = element_text(size = 13, face = "bold", hjust = 0),
        plot.tag.position = c(0, 1))

figure_2

In [ ]:
ggsave("figure_2.pdf",
       figure_2,
       width     = 460,    
       height    = 460,
       units     = "mm",
       limitsize = FALSE,
       device    = cairo_pdf)

## 4. Longitudinal Δ diversity + linear mixed-effects models


In [ ]:
tp1_meta_delta_alpha <- tp1_meta_alpha %>% 
                        filter(Studienummer %in% tp2_meta_alpha$Studienummer)
tp1_meta_delta_alpha<- tp1_meta_delta_alpha %>% mutate(Time_point = rep(c("TP1"), times = c(length(shannon))))
head(tp1_meta_delta_alpha)

In [ ]:
tp2_meta_delta_alpha <- tp2_meta_alpha %>% mutate(Time_point = rep(c("TP2"), times = c(length(shannon))))
head(tp2_meta_delta_alpha)

In [ ]:
names(tp1_meta_delta_alpha)[names(tp1_meta_delta_alpha) == "kit1.faecal_sample.barcode"] <- "Sample_ID"

names(tp2_meta_delta_alpha)[names(tp2_meta_delta_alpha) == "kit2.faecal_sample.barcode"] <- "Sample_ID"

In [ ]:
all_delta_alpha <- bind_rows(tp1_meta_delta_alpha, tp2_meta_delta_alpha)
dim(all_delta_alpha)


In [ ]:
all_delta_alpha <- all_delta_alpha %>% select (-c(kit1.faecal_sample.barcode, kit2.faecal_sample.barcode))
head(all_delta_alpha)

In [ ]:

all_delta_alpha$Time_point <- factor(all_delta_alpha$Time_point, levels = c("TP1", "TP2"))


### Adjusted

In [ ]:
model_shan_tp1_tp2 <- lmer(shannon ~ Group * Time_point + age_checked + 
                           BMI_prior + Primipara + Q1_healthydiet + (1 | Studienummer),
  data = all_delta_alpha ,
  REML = TRUE
)

summary(model_shan_tp1_tp2)

In [ ]:
coefs_shan <- summary(model_shan_tp1_tp2)$coefficients
group_p_shan <- coefs_shan["GroupCase:Time_pointTP2", "Pr(>|t|)"]
p_label_shan <- paste0("p = ", formatC(group_p_shan, format = "f", digits = 3))
p_label_shan

In [ ]:
model_rich_tp1_tp2 <- lmer(Richness ~ Group * Time_point + age_checked + 
                           BMI_prior + Primipara + Q1_healthydiet + (1 | Studienummer),
  data = all_delta_alpha,
  REML = TRUE
)
summary(model_rich_tp1_tp2)

In [ ]:
coefs_rich <- summary(model_rich_tp1_tp2)$coefficients
group_p_rich <- coefs_rich["GroupCase:Time_pointTP2", "Pr(>|t|)"]
p_label_rich <- paste0("p = ", formatC(group_p_rich , format = "f", digits = 3))
p_label_rich

In [ ]:
model_Eveness_tp1_tp2 <- lmer(Evenness ~ Group * Time_point + age_checked + 
                           BMI_prior + Primipara + Q1_healthydiet + (1 | Studienummer),
  data = all_delta_alpha,
  REML = TRUE
)

summary(model_Eveness_tp1_tp2)

In [ ]:
coefs_even <- summary(model_Eveness_tp1_tp2)$coefficients
group_p_coefs_even <- coefs_even["GroupCase:Time_pointTP2", "Pr(>|t|)"]
p_label_even <- paste0("p = ", formatC(group_p_coefs_even , format = "f", digits = 3))
p_label_even

## 5. Raincloud plots (Figure 3)


In [ ]:
plot_1 <- ggplot(
 all_delta_alpha[all_delta_alpha$Time_point %in% c('TP1', 'TP2'), ], 
  aes(x = Time_point, y = shannon, fill = Group, color = Group)) +
  geom_rain(
    alpha = 0.5, 
    rain.side = 'f2x2', 
    id.long.var = "Studienummer", 
    violin.args = list(color = NA, alpha = 0.7))+
 
  theme_classic() +
  scale_fill_manual(values = c("Control" = "#40E0D0", "Case" = "deeppink")) +
  scale_color_manual(values = c("Control" = "#40E0D0", "Case" = "deeppink"), name = "Group") +
  labs(
    x = "Time Point",
    y = "Shannon Diversity ",
    title = "(A) Shannon Diversity "
  ) +
 theme(
    legend.title = element_text(size = 14, color = "black", face = "bold"),  
    legend.text = element_text(size = 14, color = "black"),   
    axis.title.x = element_text(size = 15, color = "black",face = "bold"), 
    axis.title.y = element_text(size = 15, color = "black",face = "bold"), 
    axis.text.x = element_text(size = 14, color = "black"),  
    axis.text.y = element_text(size = 14, color = "black"),
      plot.title = element_text(size = 16, color = "black", face = "bold")  
  )

y_top <- max(all_delta_alpha$shannon, na.rm = TRUE)
plot_1 <- plot_1 +
  annotate("segment", x = 1, xend = 2, y = y_top * 1.02, yend = y_top * 1.02,
           color = "black") +                                   
  annotate("text", x = 1.5, y = y_top * 1.04,
           label = p_label_shan, size = 5, face = "bold") +    
  coord_cartesian(clip = "off") +                               
  scale_y_continuous(expand = expansion(mult = c(0.05, 0.12)))  

plot_1


In [ ]:
plot_2 <- ggplot(
all_delta_alpha[all_delta_alpha$Time_point %in% c('TP1', 'TP2'), ], 
  aes(x = Time_point, y = Richness, fill = Group, color = Group)) +
  geom_rain(
    alpha = 0.5, 
    rain.side = 'f2x2', 
    id.long.var = "Studienummer", 
    violin.args = list(color = NA, alpha = 0.7))+
 
  theme_classic() +
  scale_fill_manual(values = c("Control" = "#40E0D0", "Case" = "deeppink")) +
  scale_color_manual(values = c("Control" = "#40E0D0", "Case" = "deeppink"), name = "Group") +
  labs(
    x = "Time Point",
    y = "Richness",
    title = "(B) Observed Species"
  ) +
 theme(
    legend.title = element_text(size = 14, color = "black", face = "bold"),  
    legend.text = element_text(size = 14, color = "black", face = "bold"),   
    axis.title.x = element_text(size = 15, color = "black",face = "bold"), 
    axis.title.y = element_text(size = 15, color = "black", face = "bold"), 
    axis.text.x = element_text(size = 14, color = "black"),  
    axis.text.y = element_text(size = 14, color = "black"),
      plot.title = element_text(size = 16, color = "black", face = "bold")  
  )
y_top <- max(all_delta_alpha$Richness, na.rm = TRUE)
plot_2 <- plot_2 +
  annotate("segment", x = 1, xend = 2, y = y_top * 1.02, yend = y_top * 1.02,
           color = "black") +                                   
  annotate("text", x = 1.5, y = y_top * 1.04,
           label = p_label_rich, size = 5, face = "bold") +    
  coord_cartesian(clip = "off") +                               
  scale_y_continuous(expand = expansion(mult = c(0.05, 0.12)))  

plot_2
 

In [ ]:
plot_3 <- ggplot(
all_delta_alpha[all_delta_alpha$Time_point %in% c('TP1', 'TP2'), ], 
  aes(x = Time_point, y = Evenness, fill = Group, color = Group)) +
  geom_rain(
    alpha = 0.5, 
    rain.side = 'f2x2', 
    id.long.var = "Studienummer", 
    violin.args = list(color = NA, alpha = 0.7))+
 
  theme_classic() +
  scale_fill_manual(values = c("Control" = "#40E0D0", "Case" = "deeppink")) +
  scale_color_manual(values = c("Control" = "#40E0D0", "Case" = "deeppink"), name = "Group") +
  labs(
    x = "Time Point",
    y = "Evenness",
    #title = "(C) Pielous Evenness by Time Point and Group"
      title = "(C) Pielous Evenness"
  ) +
 theme(
    legend.title = element_text(size = 14, color = "black", face = "bold"),  
    legend.text = element_text(size = 14, color = "black"),   
    axis.title.x = element_text(size = 15, color = "black", face = "bold"), 
    axis.title.y = element_text(size = 15, color = "black", face = "bold"),
    axis.text.x = element_text(size = 14, color = "black"),  
    axis.text.y = element_text(size = 14, color = "black"),
      plot.title = element_text(size = 16, color = "black", face = "bold")  
  )
y_top <- max(all_delta_alpha$Evenness, na.rm = TRUE)
plot_3 <- plot_3 +
  annotate("segment", x = 1, xend = 2, y = y_top * 1.02, yend = y_top * 1.02,
           color = "black") +                                   
  annotate("text", x = 1.5, y = y_top * 1.04,
           label = p_label_even, size = 5, face = "bold") +    
  coord_cartesian(clip = "off") +                               
  scale_y_continuous(expand = expansion(mult = c(0.05, 0.12)))  

plot_3

In [ ]:
Figure_3 <- (plot_1 | plot_2) / plot_3 + 
  plot_layout(guides = 'collect') +           
  theme(
    legend.position = "bottom",                
    legend.title = element_text(size = 16, face = "bold"),
    legend.text = element_text(size = 14)
  )

Figure_3


In [ ]:
ggsave("Figure_3_final.pdf",
       Figure_3,
       width     = 460,    
       height    = 460,
       units     = "mm",
       limitsize = FALSE,
       device    = cairo_pdf)

In [ ]:
# TIFF 
ggsave("figure_3.tiff",
       Figure_3,
       width       = 460,
       height      = 460,
       units       = "mm",
       limitsize   = FALSE,
       device      = "tiff",
       dpi         = 300,
       compression = "lzw")

# EPS
ggsave("figure_3.eps",
       Figure_3,
       width     = 460,
       height    = 460,
       units     = "mm",
       limitsize = FALSE,
       device    = cairo_ps)